In [2]:
import os
import sys
from pathlib import Path
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark = SparkSession.builder \
    .appName("Temperature Dataset RDD Lab") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Spark started successfully!")

Spark version: 4.2.0
Spark started successfully!


In [3]:
from pathlib import Path


file_path = str(Path.cwd() / "temperature.txt")
data = sc.textFile(file_path)


records = (
    data.filter(lambda line: line.strip() != "")
        .map(lambda line: [field.strip() for field in line.split(",")])
        .filter(lambda row: len(row) == 6)
        .filter(lambda row: row[1].lower() != "average temperature")
        .map(lambda row: (row[2], float(row[1])))
)

print("Total records:", records.count())
print("Sample records:", records.take(5))

Total records: 12
Sample records: [('Bangalore', 28.5), ('Bangalore', 30.0), ('Bangalore', 27.2), ('Chennai', 32.0), ('Chennai', 29.0)]


In [8]:
#1.Maximum and minimum temperature of all cities
max_temp = records.reduceByKey(
    lambda a, b: max(a, b)
)

min_temp = records.reduceByKey(
    lambda a, b: min(a, b)
)

print("Maximum temperatures:", max_temp.collect())
print("Minimum temperatures:", min_temp.collect())

Maximum temperatures: [('Mumbai', 26.5), ('Delhi', 20.5), ('Bangalore', 30.0), ('Chennai', 34.5)]
Minimum temperatures: [('Mumbai', 24.0), ('Delhi', 16.5), ('Bangalore', 27.2), ('Chennai', 29.0)]


In [9]:
#2.Count data points for each city
result = records.map(lambda x: (x[0], 1)) \
                .reduceByKey(lambda a, b: a + b)

print(result.collect())

[('Mumbai', 3), ('Delhi', 3), ('Bangalore', 3), ('Chennai', 3)]


In [10]:
#3.Maximum and minimum temperature for Bangalore
bangalore = records.filter(lambda x: x[0].lower() == "bangalore") \
                   .map(lambda x: x[1])

print("Maximum:", bangalore.max())
print("Minimum:", bangalore.min())

Maximum: 30.0
Minimum: 27.2


In [12]:
#4.Maximum and minimum temperature for a city passed through the command line

city = input("Enter city name: ").strip().title()

city_data = records.filter(lambda x: x[0].lower() == city.lower())

if city_data.isEmpty():
    print("City not found!")
else:
    max_temp = city_data.map(lambda x: x[1]).max()
    min_temp = city_data.map(lambda x: x[1]).min()

    print("City:", city)
    print("Maximum Temperature:", max_temp)
    print("Minimum Temperature:", min_temp)

Enter city name:  Mumbai


City: Mumbai
Maximum Temperature: 26.5
Minimum Temperature: 24.0
